**Training of Computer Vision Model**

In [4]:
!pip install ultralytics


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from pathlib import Path

root = Path("../../data/raw_data/final_unified_dataset")
print("Dataset exists:", root.exists())
print("Train images:", (root / "images/train").exists())
print("Val images:", (root / "images/val").exists())
print("Test images:", (root / "images/test").exists())
print("Train labels:", (root / "labels/train").exists())
print("Val labels:", (root / "labels/val").exists())
print("Test labels:", (root / "labels/test").exists())
print("YAML exists:", (root / "dataset.yaml").exists())
print(Path("../../data/raw_data/final_unified_dataset/dataset.yaml").read_text())

Dataset exists: True
Train images: True
Val images: True
Test images: True
Train labels: True
Val labels: True
Test labels: True
YAML exists: True
path: C:\Users\HP Victus\CDS_2026Spring_project\final_unified_dataset
train: images/train
val: images/val
test: images/test

names:
  0: person
  1: bag


**Experimental Models**

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device = "mps"
)
metrics = model.val()
print(metrics)

In [ ]:
model = YOLO("yolov8s.pt") 

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=8,
    device="mps",
    workers=4,
    patience=20,
    project="yolov8s"
)

In [ ]:
model = YOLO("yolov8n.pt") 

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,
    imgsz=640,
    batch=10,
    device="mps",
    workers=2,
    patience=20,
    project="yolov8n"
)


In [ ]:
#Checking for class distribution
import os
from collections import Counter

label_dir = "../../data/raw_data/final_unified_dataset/labels/train"
class_counts = Counter()

for fname in os.listdir(label_dir):
    if not fname.endswith(".txt"):
        continue
    with open(os.path.join(label_dir, fname)) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

print("Class distribution in val labels:")
for cls_id, count in sorted(class_counts.items()):
    print(f"  Class {cls_id}: {count} instances")

In [ ]:
#Trying out a newer model

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo11n",
)

*Re-running YOLOv11 on the updated dataset with more people*

In [8]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="../../data/raw_data/final_unified_dataset/dataset.yaml",
    epochs=20,              
    imgsz=640,
    batch=10,
    device="mps",
    workers=0,
    patience=20,
    project="yolo11n_Updated_Dataset",
    cache =True
)

Ultralytics 8.4.21 🚀 Python-3.13.7 torch-2.10.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../data/raw_data/final_unified_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspect

KeyboardInterrupt: 

In [6]:
import os
from collections import Counter

base = "../../data/raw_data/final_unified_dataset"

for split in ["train", "val", "test"]:
    label_dir = os.path.join(base, "labels", split)
    counter = Counter()
    for file in os.listdir(label_dir):
        if file.endswith(".txt"):
            with open(os.path.join(label_dir, file)) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counter[int(parts[0])] += 1
    print(f"{split}: person={counter[0]}, bag={counter[1]}, ratio={counter[1]/counter[0]:.1f}x")

train: person=10761, bag=3078, ratio=0.3x
val: person=789, bag=651, ratio=0.8x
test: person=830, bag=393, ratio=0.5x


In [ ]:
import torch

print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

In [ ]:
import torch
import platform

print(f"OS:               {platform.system()}")
print(f"PyTorch version:  {torch.__version__}")
print(f"MPS available:    {torch.backends.mps.is_available()}")
print(f"CPU count:        {torch.get_num_threads()}")

# Test if multiprocessing actually works
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
    print("Multiprocessing: spawn method set successfully")
except RuntimeError as e:
    print(f"Multiprocessing warning: {e}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch
import time

# Create a dummy dataset
dummy = TensorDataset(torch.randn(100, 3, 640, 640))

# Test with workers=0
loader0 = DataLoader(dummy, batch_size=10, num_workers=0)
start = time.time()
for batch in loader0:
    pass
time0 = time.time() - start

# Test with workers=2
loader2 = DataLoader(dummy, batch_size=10, num_workers=2)
start = time.time()
for batch in loader2:
    pass
time2 = time.time() - start

print(f"workers=0: {time0:.2f}s")
print(f"workers=2: {time2:.2f}s")
print(f"Speedup:   {time0/time2:.2f}x")

if time2 < time0:
    print("Workers ARE helping")
else:
    print("Workers are NOT helping — MPS multiprocessing issue")